# 05. Simulating the storm

Optional, and not on the schedule. This part simulates overland flow during the peak of the 2 August storm with *r.sim.water*, once on the ground as the fire left it and once on the same ground before the fire. Then it compares where the water concentrates with the significant lowering from Part 03.

The roughness and infiltration values below are illustrative, not calibrated. Read the result as a comparison between the two runs, not as a reconstruction of the flood.

In [ ]:
import io
import subprocess

from workshop_utils import gj, start

tools, session = start("05", required=["dtm_2022", "dtm_2022_relief", "sig_slope"])
# Simulate at 10 m, the resolution of the land cover and burn severity.
tools.g_region(region="focal@PERMANENT", res=10, flags="a")

## Land cover before the fire

ESA WorldCover maps land cover at 10 m for 2021, a year before the fire. Read the tile that covers the site straight from its cloud-optimized GeoTIFF.

In [ ]:
WORLDCOVER = (
    "/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
    "ESA_WorldCover_10m_2021_v200_N39W123_Map.tif"
)
tools.r_import(
    input=WORLDCOVER,
    output="landcover",
    extent="region",
    resolution="region",
    resample="nearest",
)
tools.r_category(
    map="landcover",
    separator=":",
    rules=io.StringIO(
        "10:Tree cover\n20:Shrubland\n30:Grassland\n40:Cropland\n50:Built-up\n"
        "60:Bare or sparse vegetation\n80:Permanent water\n90:Herbaceous wetland\n"
    ),
)
print(tools.r_stats(input="landcover", flags="lpn").text)

## Roughness and infiltration

Each land cover class gets a Manning's n and an infiltration rate for unburned ground. Burned ground lost litter and duff, and severe burns can leave soil that repels water, so both values drop with burn severity. Change the numbers and rerun the cells below to see how much they matter.

In [ ]:
# Illustrative values for unburned ground: (Manning's n, infiltration in mm/h).
unburned = {
    10: (0.40, 30.0),  # tree cover
    20: (0.30, 25.0),  # shrubland
    30: (0.20, 20.0),  # grassland
    50: (0.02, 2.0),  # built-up
    60: (0.05, 10.0),  # bare or sparse vegetation
    80: (0.04, 0.0),  # water
}
other = (0.20, 20.0)
# Illustrative multipliers by burn severity class: (Manning's n, infiltration).
severity = {1: (1.0, 1.0), 2: (0.8, 0.8), 3: (0.5, 0.5), 4: (0.3, 0.2)}


def by_class(index):
    expression = str(other[index])
    for code, values in unburned.items():
        expression = f"if(landcover == {code}, {values[index]}, {expression})"
    return expression


def by_severity(index):
    expression = "1.0"
    for code, factors in severity.items():
        expression = (
            f"if(burn_severity@PERMANENT == {code}, {factors[index]}, {expression})"
        )
    return f"if(isnull(burn_severity@PERMANENT), 1.0, {expression})"


tools.r_mapcalc(expression=f"man_prefire = {by_class(0)}")
tools.r_mapcalc(expression=f"infil_prefire = {by_class(1)}")
tools.r_mapcalc(expression=f"man_postfire = man_prefire * {by_severity(0)}")
tools.r_mapcalc(expression=f"infil_postfire = infil_prefire * {by_severity(1)}")
for name in ("man_prefire", "man_postfire", "infil_prefire", "infil_postfire"):
    stats = tools.r_univar(map=name, format="json")
    print(f"{name:15} mean {stats['mean']:.2f}")

## The storm

Curtis et al. (2025) report rainfall rates above 200 mm/h over the Humbug and Vesa Creek watersheds and put the recurrence interval of a 100 mm/h 15-minute intensity at more than 1000 years. Hold 100 mm/h for 30 minutes as the peak of the storm. *r.sim.water* routes rainfall excess, the rain that does not soak in, so subtract the infiltration rate from the rainfall first, then run both surfaces with the same random seed.

In [ ]:
tools.r_resamp_stats(input="dtm_2022", output="dtm_10m", method="average")
tools.r_slope_aspect(elevation="dtm_10m", dx="dx_10m", dy="dy_10m")

# GRASS 8.5 calls the simulated time niterations, later versions call it duration.
interface = subprocess.run(
    ["r.sim.water", "--interface-description"], capture_output=True, text=True
).stdout
minutes = "duration" if 'name="duration"' in interface else "niterations"

rain = 100  # mm/h
for scenario in ("prefire", "postfire"):
    # r.sim.water routes rainfall excess, so take the infiltration off first.
    tools.r_mapcalc(expression=f"excess_{scenario} = max({rain} - infil_{scenario}, 0)")
    tools.r_sim_water(
        elevation="dtm_10m",
        dx="dx_10m",
        dy="dy_10m",
        rain=f"excess_{scenario}",
        man=f"man_{scenario}",
        depth=f"depth_{scenario}",
        discharge=f"discharge_{scenario}",
        random_seed=1,
        nprocs=4,
        **{minutes: 30},
    )

print(f"{'':10}{'mean depth mm':>15}{'peak discharge m3/s':>21}")
for scenario in ("prefire", "postfire"):
    depth = tools.r_univar(map=f"depth_{scenario}", format="json")
    discharge = tools.r_univar(map=f"discharge_{scenario}", format="json")
    print(f"{scenario:10}{1000 * depth['mean']:>15.1f}{discharge['max']:>21.2f}")

After the fire the peak discharge is about five times higher, while the mean depth barely changes. Smoother burned ground drains faster, so the water gathers in the channels instead of spreading across the slopes.

In [ ]:
# | label: fig-discharge
# | layout-ncol: 2
# | fig-subcap:
# |   - "Before the fire"
# |   - "After the fire"
# One fixed color scale in m3/s for both runs, so the maps compare directly.
# Cells below 0.05 m3/s are left out so the relief shows through.
scale = io.StringIO(
    "0.05 198:219:239\n0.5 107:174:214\n2 33:113:181\n10 8:48:107\n1000 8:48:107\n"
)
rules = scale.getvalue()
for scenario in ("prefire", "postfire"):
    tools.r_mapcalc(
        expression=f"flow_{scenario} = if(discharge_{scenario} >= 0.05, "
        f"discharge_{scenario}, null())"
    )
    tools.r_colors(map=f"flow_{scenario}", rules=io.StringIO(rules))
    flow = gj.Map(width=500)
    flow.d_rast(map="dtm_2022_relief")
    flow.d_rast(map=f"flow_{scenario}")
    flow.d_vect(
        map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
    )
    flow.show()

In [ ]:
# One legend for both maps above, on the same scale.
legend = gj.Map(width=1000, height=110)
legend.d_legend(
    raster="flow_postfire",
    title="Discharge (m3/s)",
    range=(0.05, 25),
    at=(35, 65, 5, 95),
    flags="stl",
    fontsize=14,
    title_fontsize=16,
)
legend.show()

## Watch it run

Run both surfaces again with `-t`, which writes the discharge every 2 minutes, and play the frames over the shaded relief. In a running notebook, drag the slider or press play. The website shows the same frames as the two animations below, which play on their own.

In [ ]:
# | output: false
frames, labels = {}, {}
for scenario in ("prefire", "postfire"):
    tools.g_remove(type="raster", pattern=f"q_{scenario}.*", flags="f")
    tools.r_sim_water(
        elevation="dtm_10m",
        dx="dx_10m",
        dy="dy_10m",
        rain=f"excess_{scenario}",
        man=f"man_{scenario}",
        discharge=f"q_{scenario}",
        output_step=2,
        random_seed=1,
        nprocs=4,
        flags="t",
        **{minutes: 30},
    )
    # Output steps can land a minute off, so read the names r.sim.water wrote.
    names = tools.g_list(type="raster", pattern=f"q_{scenario}.*").text.split()
    names.sort(key=lambda name: int(name.rsplit(".", 1)[1]))
    frames[scenario], labels[scenario] = [], []
    for source in names:
        minute = int(source.rsplit(".", 1)[1])
        frame = f"frame_{scenario}_{minute:02d}"
        tools.r_mapcalc(
            expression=f'{frame} = if("{source}" >= 0.05, "{source}", null())'
        )
        tools.r_colors(map=frame, rules=io.StringIO(rules))
        frames[scenario].append(frame)
        labels[scenario].append(f"{scenario}, {minute} min")


def storm_series(scenario):
    series = gj.SeriesMap(width=600, saved_region="focal@PERMANENT")
    series.d_rast(map="dtm_2022_relief")
    series.add_rasters(frames[scenario])
    series.add_vectors(
        ["mckinney_perimeter@PERMANENT"] * len(frames[scenario]),
        fill_color="none",
        color="black",
        width=2,
    )
    series.add_names(labels[scenario])
    return series


storm_series("postfire").show()

In [ ]:
import base64

from IPython.display import HTML, display

# Save both runs as GIFs, which also play on the workshop website, and show them
# side by side. They share the discharge legend above.
images = []
for scenario in ("prefire", "postfire"):
    storm_series(scenario).save(
        f"storm_{scenario}.gif", duration=400, text_color="black"
    )
    data = base64.b64encode(open(f"storm_{scenario}.gif", "rb").read()).decode()
    images.append(
        f'<figure style="flex: 1; margin: 0;"><img src="data:image/gif;base64,{data}" '
        f'style="width: 100%;"><figcaption>{scenario}</figcaption></figure>'
    )
display(HTML(f'<div style="display: flex; gap: 1rem;">{"".join(images)}</div>'))

## Where the water went and where the ground dropped

The DoD spans nine years, so it cannot isolate this storm. Still, if the storm did much of the work, significant lowering should be stronger where the simulated flow was high. Split the region at the 95th percentile of post-fire discharge and compare the lowering per hectare.

In [ ]:
text = tools.r_quantile(input="discharge_postfire", percentiles=95).text
threshold = float(text.strip().split(":")[-1])

tools.g_region(region="focal@PERMANENT")
tools.r_mapcalc(expression=f"flow_zone = if(discharge_postfire >= {threshold}, 1, 2)")
tools.r_category(
    map="flow_zone", separator=":", rules=io.StringIO("1:high flow\n2:rest\n")
)
tools.r_mapcalc(expression="lowering = if(sig_slope < 0, sig_slope, null())")
area = {
    z["zone"]: z["n"]
    for z in tools.r_univar(map="dtm_2022", zones="flow_zone", format="json")
}
for zone in tools.r_univar(map="lowering", zones="flow_zone", format="json"):
    hectares = area[zone["zone"]] / 10_000
    print(
        f"{zone['zone_label']:10} {hectares:7.1f} ha  "
        f"{zone['sum'] / hectares:8.0f} m3/ha of significant lowering"
    )

The cells in the top 5% of simulated flow lost about 1,150 m3 per hectare to significant lowering, against about 720 elsewhere. That fits a storm that cut into the channels, though the nine-year DoD cannot rule out other causes.

## Erosion and deposition

*r.sim.sediment* runs the same SIMWE walkers on the water depth from *r.sim.water* and moves sediment where the flow can detach and carry it. Use the same illustrative coefficients for both runs, so any difference between them comes from the flow the fire changed. SIMWE reports erosion as negative and deposition as positive, in kg/m2/s.

In [ ]:
tools.g_region(region="focal@PERMANENT", res=10, flags="a")
# Illustrative coefficients: detachment (s/m), transport (s), critical shear stress (Pa).
tools.r_mapcalc(expression="detachment = 0.001")
tools.r_mapcalc(expression="transport = 0.001")
tools.r_mapcalc(expression="shear = 0.0")

for scenario in ("prefire", "postfire"):
    tools.r_sim_sediment(
        elevation="dtm_10m",
        water_depth=f"depth_{scenario}",
        dx="dx_10m",
        dy="dy_10m",
        detachment_coeff="detachment",
        transport_coeff="transport",
        shear_stress="shear",
        man=f"man_{scenario}",
        erosion_deposition=f"erdep_{scenario}",
        sediment_flux=f"sedflux_{scenario}",
        random_seed=1,
        nprocs=4,
        **{minutes: 30},
    )

print(f"{'':10}{'eroding cells':>15}{'depositing cells':>18}{'peak flux kg/m/s':>18}")
for scenario in ("prefire", "postfire"):
    tools.r_mapcalc(expression=f"tmp_erode = if(erdep_{scenario} < 0, 1, null())")
    tools.r_mapcalc(expression=f"tmp_deposit = if(erdep_{scenario} > 0, 1, null())")
    eroding = tools.r_univar(map="tmp_erode", format="json")["n"]
    depositing = tools.r_univar(map="tmp_deposit", format="json")["n"]
    flux = tools.r_univar(map=f"sedflux_{scenario}", format="json")["max"]
    print(f"{scenario:10}{eroding:>15,}{depositing:>18,}{flux:>18.3f}")

In [ ]:
# | label: fig-erdep
# | layout-ncol: 2
# | fig-subcap:
# |   - "Before the fire"
# |   - "After the fire"
# Red for erosion, blue for deposition, on one scale set by the post-fire run.
text = tools.r_quantile(input="erdep_postfire", percentiles="2,98").text
low, high = (abs(float(line.split(":")[-1])) for line in text.strip().splitlines())
limit = max(low, high)
erdep_rules = (
    f"-1000 103:0:31\n{-limit} 178:24:43\n0 247:247:247\n"
    f"{limit} 33:102:172\n1000 5:48:97\n"
)
for scenario in ("prefire", "postfire"):
    tools.r_colors(map=f"erdep_{scenario}", rules=io.StringIO(erdep_rules))
    view = gj.Map(width=500)
    view.d_shade(shade="dtm_2022_relief", color=f"erdep_{scenario}")
    view.d_vect(
        map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
    )
    view.show()

In [ ]:
# One legend for both maps above, on the same scale.
legend = gj.Map(width=1000, height=110)
legend.d_legend(
    raster="erdep_postfire",
    title="Erosion (-) and deposition (+), kg/m2/s",
    range=(-limit, limit),
    at=(35, 65, 5, 95),
    flags="st",
    fontsize=14,
    title_fontsize=16,
)
legend.show()

Compare with the lidar. Take the 10% of cells where SIMWE predicts the most erosion after the fire and the 10% where it predicts the most deposition, and total the significant change from Part 03 in each.

In [ ]:
text = tools.r_quantile(input="erdep_postfire", percentiles="10,90").text
p10, p90 = (float(line.split(":")[-1]) for line in text.strip().splitlines())

tools.g_region(region="focal@PERMANENT")
tools.r_mapcalc(
    expression=f"simwe_zone = if(erdep_postfire <= {p10}, 1, "
    f"if(erdep_postfire >= {p90}, 2, 3))"
)
tools.r_category(
    map="simwe_zone",
    separator=":",
    rules=io.StringIO("1:most erosion\n2:most deposition\n3:rest\n"),
)
area = {
    z["zone"]: z["n"]
    for z in tools.r_univar(map="dtm_2022", zones="simwe_zone", format="json")
}
for zone in tools.r_univar(map="sig_slope", zones="simwe_zone", format="json"):
    hectares = area[zone["zone"]] / 10_000
    print(
        f"{zone['zone_label']:16} {hectares:7.1f} ha  "
        f"{zone['sum'] / hectares:+8.0f} m3/ha of significant change"
    )

The lidar shows lowering in all three groups. The cells SIMWE marks for the most erosion lost a little more, about 720 m3 per hectare against about 600 for the rest, but the cells it marks for deposition did not gain. The broad lowering after the fire dominates the nine-year DoD, and the coefficients are not calibrated, so this comparison cannot confirm the simulated pattern.

## Limits

- Steady rain at the peak intensity, not the storm's three-hour hyetograph.
- Roughness and infiltration are illustrative values. Calibrating them needs field measurements or a runoff record at the outlet.
- A 10 m grid smooths the smallest channels.
- Erosion and deposition use one set of illustrative coefficients for the whole region.

## Your turn

Set `rain = 200`, the rate Curtis et al. report in isolated cells, and rerun, or change the infiltration multiplier for high severity, and watch the peak discharge.